# Session 10: MongoDB Tools: Compass and BI Connector

Diploma in AI · companion notebook

Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom. In Colab, the first cell installs anything that is missing.

In [1]:
import os, sys, subprocess
try:
    import google.colab  # noqa
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mongomock', 'pymongo<4.9'], check=False)
    # fetch helper/data files used by the examples
    base = 'https://raw.githubusercontent.com/Hamnakhan123/Diploma-in-AI/main/term-1/02-managing-large-datasets-with-mongodb/code/'
    for f in ['store_db.py']:
        if not os.path.exists(f):
            subprocess.run(['wget', '-q', base + f], check=False)
print('ready')

ready


## Example: `session-10_import_export.py`

In [2]:
# Session 10 · Doing what mongoexport / mongoimport do, from Python
import csv, json, os, tempfile
from datetime import datetime
from bson import json_util
from store_db import get_db

def ext_json(v):
    """Encode BSON types the way mongoexport writes Extended JSON (relaxed mode)."""
    if isinstance(v, datetime):
        return {"$date": v.strftime("%Y-%m-%dT%H:%M:%S.000Z")}
    raise TypeError(type(v))

db = get_db()
HOME = os.getcwd()
os.chdir(tempfile.mkdtemp())                     # work in a scratch folder
os.makedirs("exports", exist_ok=True)

# mongoexport --db kgs --collection customers --out exports/customers.json   (one JSON doc per line)
with open("exports/customers.json", "w") as f:
    for d in db.customers.find():
        f.write(json.dumps(d, default=ext_json) + "\n")   # Extended JSON keeps BSON types like dates
print("customers.json (first 2 lines):")
print("".join(open("exports/customers.json").readlines()[:2]))

# mongoexport --type=csv --fields name,category,price,specs.brand --out exports/products.csv
fields = ["_id", "name", "category", "price", "specs.brand"]
with open("exports/products.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(fields)
    for d in db.products.find():
        w.writerow([d["_id"], d["name"], d["category"], d["price"], d["specs"]["brand"]])
print("products.csv (first 4 lines):")
print("".join(open("exports/products.csv").readlines()[:4]))

# mongoimport --type=csv --headerline --columnsHaveTypes  (types must be converted on import)
with open("exports/new_products.csv", "w") as f:
    f.write("_id,name,category,price,stock\n112,Smart Watch,Wearables,24000,7\n113,Fitness Band,Wearables,6500,15\n")
with open("exports/new_products.csv") as f:
    rows = [{**r, "_id": int(r["_id"]), "price": int(r["price"]), "stock": int(r["stock"])} for r in csv.DictReader(f)]
db.products.insert_many(rows)
print("imported", len(rows), "rows; Wearables now:",
      [d["name"] for d in db.products.find({"category": "Wearables"})])

# Round trip: re-import the JSON export into a new collection (mongoimport --collection customers_copy)
with open("exports/customers.json") as f:
    db.customers_copy.insert_many([json_util.loads(line) for line in f])
c = db.customers_copy.find_one({"_id": 1})
print("round-trip check: joined is still a", type(c["joined"]).__name__)
os.chdir(HOME)

customers.json (first 2 lines):
{"_id": 1, "name": "Ayesha Khan", "city": "Karachi", "email": "ayesha@example.pk", "joined": {"$date": "2025-01-12T00:00:00.000Z"}, "loyalty": "Gold"}
{"_id": 2, "name": "Bilal Ahmed", "city": "Lahore", "email": "bilal@example.pk", "joined": {"$date": "2025-02-03T00:00:00.000Z"}, "loyalty": "Silver"}

products.csv (first 4 lines):
_id,name,category,price,specs.brand
101,Wireless Mouse,Accessories,1500,Logi
102,Mechanical Keyboard,Accessories,7500,Redragon
103,27-inch Monitor,Displays,52000,Dell

imported 2 rows; Wearables now: ['Smart Watch', 'Fitness Band']
round-trip check: joined is still a datetime


## Example: `session-10_schema_analysis.py`

In [3]:
# Session 10 · What Compass's Schema tab shows, computed by hand
from collections import Counter, defaultdict
from store_db import get_db

db = get_db()

def analyse(coll):
    docs = list(coll.find())
    seen, types, values = Counter(), defaultdict(Counter), defaultdict(list)
    def walk(d, prefix=""):
        for k, v in d.items():
            path = prefix + k
            seen[path] += 1
            types[path][type(v).__name__ if v is not None else "null"] += 1
            if isinstance(v, dict):
                walk(v, path + ".")
            elif isinstance(v, (int, float)) and not isinstance(v, bool):
                values[path].append(v)
    for d in docs:
        walk(d)
    print(f"{coll.name}: {len(docs)} documents")
    print(f"  {'field':<22}{'present':>8}  types                     range")
    for path in sorted(seen):
        t = ", ".join(f"{k} {v / len(docs):.0%}" for k, v in types[path].items())
        rng = f"{min(values[path]):,} .. {max(values[path]):,}" if values[path] else ""
        print(f"  {path:<22}{seen[path] / len(docs):>8.0%}  {t:<25} {rng}")
    print()

analyse(db.products)
analyse(db.customers)

products: 10 documents
  field                  present  types                     range
  _id                       100%  int 100%                  101 .. 110
  category                  100%  str 100%                  
  name                      100%  str 100%                  
  price                     100%  int 100%                  1,500 .. 285,000
  specs                     100%  dict 100%                 
  specs.brand               100%  str 100%                  
  specs.ramGB                20%  int 20%                   16 .. 16
  specs.warrantyMonths      100%  int 100%                  12 .. 36
  stock                     100%  int 100%                  0 .. 40
  tags                      100%  list 100%                 

customers: 6 documents
  field                  present  types                     range
  _id                       100%  int 100%                  1 .. 6
  city                      100%  str 100%                  
  email                      83%  

## Example: `session-10_bi_sql.py`

In [4]:
# Session 10 · What the BI Connector does: present documents as SQL tables
# The BI Connector (and the newer Atlas SQL Interface) maps each collection to a table,
# flattens sub-documents into columns and turns arrays into child tables, so tools such as
# Power BI, Tableau or Excel can query MongoDB with SQL. This file performs the same mapping
# into SQLite and runs the kind of SQL a BI tool would send.
import sqlite3
from store_db import get_db

db = get_db()
sql = sqlite3.connect(":memory:")

sql.execute("CREATE TABLE products (_id INT, name TEXT, category TEXT, price INT, stock INT, `specs.brand` TEXT)")
sql.execute("CREATE TABLE orders (_id INT, customerId INT, orderDate TEXT, status TEXT, payment TEXT)")
sql.execute("CREATE TABLE orders_items (_id INT, items_idx INT, `items.productId` INT, `items.qty` INT, `items.price` INT)")
sql.execute("CREATE TABLE customers (_id INT, name TEXT, city TEXT, loyalty TEXT)")

for p in db.products.find():
    sql.execute("INSERT INTO products VALUES (?,?,?,?,?,?)", (p["_id"], p["name"], p["category"], p["price"], p["stock"], p["specs"]["brand"]))
for o in db.orders.find():
    sql.execute("INSERT INTO orders VALUES (?,?,?,?,?)", (o["_id"], o["customerId"], o["orderDate"].date().isoformat(), o["status"], o["payment"]))
    for i, it in enumerate(o["items"]):                  # array -> child table with an index column
        sql.execute("INSERT INTO orders_items VALUES (?,?,?,?,?)", (o["_id"], i, it["productId"], it["qty"], it["price"]))
for c in db.customers.find():
    sql.execute("INSERT INTO customers VALUES (?,?,?,?)", (c["_id"], c["name"], c["city"], c["loyalty"]))

print("Tables a BI tool would see:", [r[0] for r in sql.execute("SELECT name FROM sqlite_master")])

query = """
SELECT c.city,
       COUNT(DISTINCT o._id)                      AS orders,
       SUM(i.`items.qty` * i.`items.price`)       AS revenue
FROM orders o
JOIN orders_items i ON i._id = o._id
JOIN customers   c ON c._id = o.customerId
WHERE o.status = 'Delivered'
GROUP BY c.city
ORDER BY revenue DESC;
"""
print("\nSQL sent by a dashboard:\n" + query.strip())
print("\n{:<10} {:>6} {:>10}".format("city", "orders", "revenue"))
for city, n, rev in sql.execute(query):
    print("{:<10} {:>6} {:>10,}".format(city, n, rev))

Tables a BI tool would see: ['products', 'orders', 'orders_items', 'customers']

SQL sent by a dashboard:
SELECT c.city,
       COUNT(DISTINCT o._id)                      AS orders,
       SUM(i.`items.qty` * i.`items.price`)       AS revenue
FROM orders o
JOIN orders_items i ON i._id = o._id
JOIN customers   c ON c._id = o.customerId
WHERE o.status = 'Delivered'
GROUP BY c.city
ORDER BY revenue DESC;

city       orders    revenue
Karachi         5    319,500
Lahore          1     52,000
